# Kaggle Playground Series S6E8
## Part 1 - Setup & Memory Optimization

In [ ]:
# ============================================================
# Libraries
# ============================================================
import os
import gc
import random
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.style.use("ggplot")


In [ ]:
# ============================================================
# Configuration
# ============================================================

SEED = 42
TARGET = "addicted_label"
ID = "id"
N_SPLITS = 5

random.seed(SEED)
np.random.seed(SEED)


## Memory Reduction Utility

In [ ]:
def reduce_memory(df):
    start_mem = df.memory_usage(deep=True).sum()/1024**2
    print(f"Memory before: {start_mem:.2f} MB")

    for col in df.columns:
        col_type = df[col].dtype

        if pd.api.types.is_integer_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="integer")

        elif pd.api.types.is_float_dtype(col_type):
            df[col] = pd.to_numeric(df[col], downcast="float")

        elif col_type == object:
            if df[col].nunique(dropna=False)/len(df) < 0.5:
                df[col] = df[col].astype("category")

    end_mem = df.memory_usage(deep=True).sum()/1024**2
    print(f"Memory after : {end_mem:.2f} MB")
    print(f"Reduced by {(start_mem-end_mem)/start_mem*100:.1f}%")
    return df


## Load Dataset

**Kaggle paths** are enabled below. Uncomment the local paths if running on your PC.

In [ ]:
# Kaggle
train = pd.read_csv('/kaggle/input/playground-series-s6e8/train.csv')
test = pd.read_csv('/kaggle/input/playground-series-s6e8/test.csv')
sample = pd.read_csv('/kaggle/input/playground-series-s6e8/sample_submission.csv')

# Local
# train = pd.read_csv('train.csv')
# test = pd.read_csv('test.csv')
# sample = pd.read_csv('sample_submission.csv')

train = reduce_memory(train)
test = reduce_memory(test)

print("Train:", train.shape)
print("Test :", test.shape)


## Dataset Overview

In [ ]:
display(train.head())
display(train.describe(include='all').T)
train.info()


## Missing Values

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]
display(missing.to_frame("Missing"))


## Feature Lists

In [ ]:
FEATURES = [c for c in train.columns if c not in [ID, TARGET]]

CAT_FEATURES = train[FEATURES].select_dtypes(
    include=['object','category']
).columns.tolist()

NUM_FEATURES = [c for c in FEATURES if c not in CAT_FEATURES]

print("Categorical Features")
print(CAT_FEATURES)

print("\nNumerical Features")
print(NUM_FEATURES)


## Target Distribution

In [ ]:
print(train[TARGET].value_counts())
print()
print(train[TARGET].value_counts(normalize=True))

plt.figure(figsize=(5,4))
sns.countplot(x=train[TARGET])
plt.title("Target Distribution")
plt.show()


> ✅ Part 1 Complete.

Next notebook will contain a full professional EDA.